# 04 — ทำนาย PM2.5 จาก AOD ด้วย Random Forest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_AOD_PM_PCD/blob/main/04_Random_Forest_PM25.ipynb)

## จุดประสงค์การเรียนรู้

เมื่อจบ Notebook นี้ นิสิตสามารถ
1. อธิบายหลักการของ **Random Forest** สำหรับงาน regression
2. สร้างและประเมินแบบจำลองด้วย **$R^2$, RMSE และ MAE**
3. อธิบายความต่างระหว่าง cross-validation แบบสุ่ม กับแบบ **กันออกทั้งสถานี** และเลือกใช้ได้ถูกกับคำถาม
4. อ่าน **variable importance** อย่างระมัดระวัง

**เวลาโดยประมาณ** 110 นาที

> **วิธีเรียน** — อ่านคำอธิบาย → รันทีละ cell → ดูผล → ตอบคำถาม "ลองคิด" ในใจก่อนรัน cell ถัดไป
> ไม่แนะนำให้กด *Run all* ในรอบแรก

## 1. หลักการ

Notebook 03 พบว่า AOD ตัวเดียวอธิบาย PM2.5 ได้เพียงบางส่วน และความสัมพันธ์ต่างกันตามพื้นที่
เราจึงต้องการแบบจำลองที่ (1) ใช้ตัวแปรได้หลายตัว และ (2) ไม่บังคับให้ความสัมพันธ์เป็นเส้นตรง

### 1.1 Random Forest ทำงานอย่างไร

```
ข้อมูลฝึก ──► สุ่มตัวอย่างซ้ำ (bootstrap) B ชุด
                 │
     ┌───────────┼───────────┐
   ต้นไม้ 1     ต้นไม้ 2  ...  ต้นไม้ B      แต่ละต้นถามคำถามเช่น "AOD > 0.6 ไหม" "lat > 17 ไหม"
     │           │           │
   42 µg/m³    55 µg/m³    47 µg/m³
     └───────────┼───────────┘
            ค่าเฉลี่ย = คำตอบ
```

$$\hat{y}(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^{B} T_b(\mathbf{x})$$

| สัญลักษณ์ | ความหมาย |
|---|---|
| $\mathbf{x}$ | ชุดตัวแปรทำนาย (predictors) ของหนึ่งสถานี-วัน |
| $T_b(\mathbf{x})$ | ค่าที่ต้นไม้ต้นที่ $b$ ทำนาย |
| $B$ | จำนวนต้นไม้ |

ต้นไม้ต้นเดียวจำข้อมูลฝึกได้ดีเกินไป (overfit) การเฉลี่ยหลายต้นที่ต่างกันเล็กน้อยช่วยลดปัญหานี้

### 1.2 ตัวแปรทำนายที่ใช้

$$\mathbf{x} = [\ \text{AOD},\ \text{Column WV},\ \text{lat},\ \text{lon},\ \text{DOY}\ ]$$

| ตัวแปร | เหตุผล |
|---|---|
| AOD (550 nm) | ปริมาณละอองลอยทั้งคอลัมน์ |
| Column WV | ไอน้ำในคอลัมน์ (cm) เกี่ยวกับความชื้นซึ่งทำให้อนุภาคพองตัว |
| lat, lon | ตำแหน่ง แทนลักษณะประจำพื้นที่ที่เราไม่ได้วัด |
| DOY | วันที่ของปี แทนการเปลี่ยนแปลงตามช่วงฤดู |

ชุดตัวแปรนี้ **เล็กโดยตั้งใจ** ทุกตัวมาจากดาวเทียมหรือรู้ได้ทุกตำแหน่ง จึงนำไปทำแผนที่ต่อได้ใน Notebook 05

### 1.3 ตัวชี้วัด

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} \qquad RMSE = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2} \qquad MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

RMSE ลงโทษความผิดพลาดขนาดใหญ่หนักกว่า MAE ถ้า RMSE มากกว่า MAE มาก แปลว่ามีบางวันที่ทำนายพลาดมาก

### 1.4 Cross-validation — ทดสอบกับข้อมูลที่แบบจำลองไม่เคยเห็น

แบ่งข้อมูลเป็น 5 ส่วน ฝึกด้วย 4 ส่วน ทดสอบกับ 1 ส่วน แล้วหมุนจนครบ คำถามคือ **แบ่งอย่างไร**

```
แบบสุ่ม (random)              : วันต่าง ๆ ของสถานีเดียวกันอยู่ทั้งฝั่งฝึกและฝั่งทดสอบ
                                 → ตอบคำถาม "เติมวันที่หายของสถานีเดิมได้ดีแค่ไหน"
แบบกันสถานี (station-based)   : สถานีที่ใช้ทดสอบไม่เคยอยู่ในชุดฝึกเลย
                                 → ตอบคำถาม "ทำนายในที่ที่ไม่มีสถานีได้ดีแค่ไหน"
```

เป้าหมายของเราคือทำแผนที่ในพื้นที่ที่ **ไม่มีสถานี** แบบที่สองจึงเป็นการทดสอบที่ตรงคำถาม

## 0. เตรียมเครื่องมือ

Cell ชุดนี้เหมือนกันทุก Notebook ทำ 3 อย่าง คือ โหลด library, ดาวน์โหลดข้อมูลจาก GitHub ของรายวิชา และสร้างฟังก์ชันแผนที่มาตรฐาน
รันทีละ cell ตามลำดับ ไม่จำเป็นต้องอ่านโค้ดส่วนนี้ละเอียดในรอบแรก

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

YEAR = 2023
DATA_URL = "https://raw.githubusercontent.com/nattaponm/Teach_AOD_PM_PCD/main/data/"
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

def fetch(name):
    # ดาวน์โหลดไฟล์จาก GitHub มาเก็บในโฟลเดอร์ data (โหลดครั้งเดียว)
    path = DATA_DIR / name
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + name, path)
    return path

In [ ]:
# รูปแบบภาพสำหรับงานตีพิมพ์: ตัวอักษรอ่านง่าย เส้นบาง ความละเอียด 300 dpi
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "legend.fontsize": 8, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
})

UNIT = "µg m$^{-3}$"

def save_fig(fig, name):
    # บันทึกทั้ง PNG (ใส่รายงาน) และ PDF (ส่งวารสาร)
    fig.savefig(FIG_DIR / f"{name}.png")
    fig.savefig(FIG_DIR / f"{name}.pdf")
    print("บันทึกแล้ว:", FIG_DIR / f"{name}.png", "และ .pdf")

In [ ]:
# ขอบเขต 77 จังหวัด และเส้นขอบประเทศ (รวมทุกจังหวัดเป็นรูปเดียว)
provinces = gpd.read_file(fetch("provinces.geojson"))
country = gpd.GeoSeries([provinces.buffer(0.002).union_all().buffer(-0.002)], crs=provinces.crs)

MAP_EXTENT = [97.0, 106.0, 5.3, 20.8]   # west, east, south, north

def base_map(ax, title=None, province_color="0.45"):
    # วาดองค์ประกอบแผนที่มาตรฐาน: จังหวัด ประเทศ พิกัด มาตราส่วน ทิศเหนือ
    provinces.boundary.plot(ax=ax, linewidth=0.25, color=province_color, zorder=3)
    country.boundary.plot(ax=ax, linewidth=0.9, color="black", zorder=4)

    ax.set_xlim(MAP_EXTENT[0], MAP_EXTENT[1]); ax.set_ylim(MAP_EXTENT[2], MAP_EXTENT[3])
    ax.set_aspect(1 / np.cos(np.deg2rad(13)))          # แก้การยืดของลองจิจูดที่ละติจูด ~13°N
    ax.set_xticks(np.arange(98, 107, 2)); ax.set_yticks(np.arange(6, 21, 2))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}°E"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}°N"))
    ax.tick_params(direction="in", top=True, right=True, length=3)
    ax.grid(linewidth=0.3, color="0.8", linestyle=":", zorder=0)

    # มาตราส่วน 200 km (1 องศาลองจิจูด = 111.32 x cos(ละติจูด) km)
    lat0, lon0 = 6.1, 102.9
    width = 200 / (111.32 * np.cos(np.deg2rad(lat0)))
    ax.plot([lon0, lon0 + width], [lat0, lat0], color="black", linewidth=1.5, zorder=5)
    ax.text(lon0 + width / 2, lat0 + 0.18, "200 km", ha="center", va="bottom", fontsize=7)

    # ลูกศรทิศเหนือ
    ax.annotate("N", xy=(105.2, 20.2), xytext=(105.2, 19.2), ha="center", va="center", fontsize=9,
                arrowprops=dict(arrowstyle="-|>", color="black", linewidth=1))
    ax.set_xlabel(""); ax.set_ylabel("")              # พิกัดบอกอยู่ที่ตัวเลขแกนแล้ว
    if title:
        ax.set_title(title)
    return ax

print("จำนวนจังหวัด:", len(provinces))

## 2. เตรียมข้อมูล

In [ ]:
m = pd.read_csv(fetch("aod_pm_matchup.csv"), parse_dates=["date"])
m["doy"] = m["date"].dt.dayofyear            # วันที่ของปี (1-365) แทนฤดูกาล

FEATURES = ["aod_055", "column_wv", "lat", "lon", "doy"]
MIN_PIXELS = 3

pairs = m.dropna(subset=["pm25"] + FEATURES)
pairs = pairs[pairs["aod_n_pixels"] >= MIN_PIXELS].reset_index(drop=True)

X = pairs[FEATURES]
y = pairs["pm25"]
groups = pairs["station"]                    # ใช้แบ่งข้อมูลแบบ "กันออกทั้งสถานี"

print("จำนวนคู่ข้อมูล:", len(pairs), "| สถานี:", groups.nunique())

## 3. สร้างแบบจำลองสองแบบ

- **Linear** — เส้นตรงจาก AOD ตัวเดียว (แบบ Notebook 03) ใช้เป็นฐานเปรียบเทียบ (baseline)
- **Random Forest** — ใช้ตัวแปรทั้ง 5 ตัว

In [ ]:
linear = LinearRegression()
forest = RandomForestRegressor(
    n_estimators=200,       # จำนวนต้นไม้ B
    min_samples_leaf=3,     # แต่ละใบต้องมีอย่างน้อย 3 ตัวอย่าง ช่วยลด overfit
    n_jobs=-1,
    random_state=42,        # กำหนดค่าเพื่อให้ได้ผลเหมือนเดิมทุกครั้งที่รัน
)

def evaluate(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

## 4. ทดสอบด้วย cross-validation สองแบบ

`cross_val_predict` คืนค่าทำนายของทุกแถว โดยแต่ละแถวถูกทำนายจากแบบจำลองที่ **ไม่ได้ฝึกด้วยแถวนั้น**

> 💭 **ลองคิด** — ก่อนรัน ทายว่า Random Forest จะได้ $R^2$ สูงกว่าในการแบ่งแบบใด และต่างกันมากหรือน้อย

In [ ]:
cv_random = KFold(n_splits=5, shuffle=True, random_state=42)
cv_station = GroupKFold(n_splits=5)

pred = {
    ("Linear (AOD only)", "random"): cross_val_predict(linear, X[["aod_055"]], y, cv=cv_random),
    ("Linear (AOD only)", "station"): cross_val_predict(linear, X[["aod_055"]], y, cv=cv_station, groups=groups),
    ("Random Forest", "random"): cross_val_predict(forest, X, y, cv=cv_random),
    ("Random Forest", "station"): cross_val_predict(forest, X, y, cv=cv_station, groups=groups),
}

results = pd.DataFrame({key: evaluate(y, p) for key, p in pred.items()}).T
results.index.names = ["model", "CV split"]
results.round(2)

อ่านตารางสองทิศทาง
- **ระหว่างแบบจำลอง** (แถว Linear เทียบ Random Forest ในการแบ่งแบบเดียวกัน): ตัวแปรเพิ่มและความไม่เป็นเส้นตรงช่วยได้เท่าใด
- **ระหว่างการแบ่ง** (random เทียบ station ในแบบจำลองเดียวกัน): ผลแบบสุ่มดูดีเกินจริงไปเท่าใด

> **หมายเหตุ** — ผลแบบกันสถานีขึ้นกับว่าสถานีใดถูกจัดอยู่กลุ่มเดียวกัน เมื่อมีเพียงไม่กี่สิบสถานี $R^2$ จึงขยับได้ราว 0.05 ระหว่างการจัดกลุ่มต่างแบบ
> และอาจต่างกันเล็กน้อยระหว่างเวอร์ชันของ scikit-learn ให้อ่านเป็น **ช่วงค่าโดยประมาณ** ไม่ใช่ตัวเลขแน่นอนถึงทศนิยมสองตำแหน่ง

### ภาพที่ 1 — ค่าที่วัดได้ เทียบกับ ค่าที่ทำนาย (แบบกันสถานี)

จุดที่อยู่บนเส้นทแยง 1:1 คือทำนายถูกพอดี

In [ ]:
lim = np.percentile(y, 99.5)
fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.3), sharex=True, sharey=True)

for ax, name in zip(axes, ["Linear (AOD only)", "Random Forest"]):
    p = pred[(name, "station")]; s = results.loc[(name, "station")]
    hb = ax.hexbin(y, p, gridsize=45, cmap="viridis", mincnt=1, bins="log", extent=(0, lim, 0, lim), linewidths=0)
    ax.plot([0, lim], [0, lim], color="black", linestyle="--", linewidth=0.8)
    ax.text(0.04, 0.96, f"R$^2$ = {s['R2']:.2f}\nRMSE = {s['RMSE']:.1f}\nMAE = {s['MAE']:.1f}",
            transform=ax.transAxes, va="top", fontsize=7.5,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=2))
    ax.set_title(name); ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal")
    ax.set_xlabel(f"Observed PM$_{{2.5}}$ ({UNIT})")
axes[0].set_ylabel(f"Predicted PM$_{{2.5}}$ ({UNIT})")
fig.colorbar(hb, ax=axes, pad=0.02, shrink=0.85, label="Number of pairs")
save_fig(fig, "fig10_observed_vs_predicted")
plt.show()

> 💭 **ลองคิด** — ที่ค่า PM2.5 สูงมาก จุดอยู่เหนือหรือใต้เส้น 1:1 แปลว่าแบบจำลองทำนายสูงไปหรือต่ำไปในวันฝุ่นหนัก

## 5. ตัวแปรใดสำคัญ

ใช้ **permutation importance**: สลับค่าของตัวแปรหนึ่งแบบสุ่ม แล้วดูว่า $R^2$ ลดลงเท่าใด ยิ่งลดมากยิ่งสำคัญ
คำนวณบนสถานีที่กันไว้ทดสอบ เพื่อให้สะท้อนการทำนายในที่ใหม่

In [ ]:
from sklearn.inspection import permutation_importance

# ใช้การแบ่งรอบแรกของ GroupKFold: ฝึกด้วย 4 ส่วน ทดสอบกับสถานีที่กันไว้ 1 ส่วน
train_idx, test_idx = next(cv_station.split(X, y, groups))
forest.fit(X.iloc[train_idx], y.iloc[train_idx])

imp = permutation_importance(forest, X.iloc[test_idx], y.iloc[test_idx], n_repeats=10, random_state=42, n_jobs=-1)
importance = pd.DataFrame({"mean": imp.importances_mean, "sd": imp.importances_std}, index=FEATURES).sort_values("mean")

LABELS = {"aod_055": "AOD (550 nm)", "column_wv": "Column water vapour", "lat": "Latitude", "lon": "Longitude", "doy": "Day of year"}

fig, ax = plt.subplots(figsize=(4.2, 2.6))
ax.barh([LABELS[f] for f in importance.index], importance["mean"], xerr=importance["sd"],
        color="#1b9e77", error_kw=dict(linewidth=0.8, capsize=2))
ax.set_xlabel("Decrease in R$^2$ when permuted")
ax.spines[["top", "right"]].set_visible(False)
save_fig(fig, "fig11_permutation_importance")
plt.show()

## 6. สรุป

```
R² จากการแบ่งแบบสุ่ม      ≠  ความแม่นยำในที่ที่ไม่มีสถานี
variable importance สูง   ≠  ตัวแปรนั้นเป็นสาเหตุ
ตัวแปรมากขึ้น             ≠  แบบจำลองดีขึ้นเสมอ
```

lat และ lon ช่วยให้แบบจำลอง "จำ" ลักษณะประจำพื้นที่ได้ ซึ่งมีประโยชน์ใกล้สถานี แต่เสี่ยงเมื่อนำไปใช้ไกลจากสถานี
ประเด็นนี้จะเห็นชัดเมื่อทำแผนที่ใน Notebook 05

---
## แบบฝึกหัด

### ข้อ 1 (ฝึกทำ) — ต้องใช้ต้นไม้กี่ต้น
วัด RMSE แบบกันสถานีของ Random Forest เมื่อ `n_estimators` เท่ากับ 10, 50 และ 200

In [ ]:
# เขียนโค้ดของคุณที่นี่
# แนวทาง: วนลูปสร้าง RandomForestRegressor(n_estimators=n, min_samples_leaf=3, n_jobs=-1, random_state=42)
#         แล้วใช้ cross_val_predict(..., cv=cv_station, groups=groups) และฟังก์ชัน evaluate

In [ ]:
#@title เฉลยข้อ 1 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
for n in [10, 50, 200]:
    model = RandomForestRegressor(n_estimators=n, min_samples_leaf=3, n_jobs=-1, random_state=42)
    p = cross_val_predict(model, X, y, cv=cv_station, groups=groups)
    print(f"{n:4d} ต้น: RMSE = {evaluate(y, p)['RMSE']:.2f} µg/m³")

<details>
<summary><b>คำอธิบายข้อ 1</b> (คลิกเพื่อเปิด)</summary>

RMSE มักลดลงเร็วในช่วงแรกแล้ว **คงที่** เมื่อจำนวนต้นไม้มากพอ การเพิ่มต้นไม้ไม่ทำให้ overfit แต่ใช้เวลาคำนวณมากขึ้น
จึงเลือกจำนวนที่ผลนิ่งแล้ว ไม่จำเป็นต้องมากที่สุด

</details>

### ข้อ 2 (ฝึกคิด) — ถ้าไม่บอกตำแหน่ง
ฝึก Random Forest โดย **ตัด lat และ lon ออก** แล้วเทียบ $R^2$ กับแบบจำลองเต็ม ทั้งการแบ่งแบบสุ่มและแบบกันสถานี

In [ ]:
# เขียนโค้ดของคุณที่นี่
# แนวทาง: features_no_loc = ["aod_055", "column_wv", "doy"]  แล้วใช้ X[features_no_loc]

my_r2_station_no_loc = None    # ใส่ค่า R2 แบบกันสถานีของแบบจำลองที่ไม่มี lat/lon

In [ ]:
#@title ตรวจคำตอบข้อ 2 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
_p = cross_val_predict(forest, X[["aod_055", "column_wv", "doy"]], y, cv=cv_station, groups=groups)
_expected = r2_score(y, _p)
if my_r2_station_no_loc is None:
    print("ยังไม่ได้ใส่คำตอบใน my_r2_station_no_loc")
elif abs(my_r2_station_no_loc - _expected) < 0.02:
    print("ถูกต้อง")
else:
    print("ยังไม่ใช่ ตรวจว่าใช้ cv=cv_station พร้อม groups=groups และตัดทั้ง lat และ lon ออก")

In [ ]:
#@title เฉลยข้อ 2 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
features_no_loc = ["aod_055", "column_wv", "doy"]
rows = {}
for label, cols in [("มี lat/lon", FEATURES), ("ไม่มี lat/lon", features_no_loc)]:
    rows[label] = {
        "R2 แบบสุ่ม": r2_score(y, cross_val_predict(forest, X[cols], y, cv=cv_random)),
        "R2 แบบกันสถานี": r2_score(y, cross_val_predict(forest, X[cols], y, cv=cv_station, groups=groups)),
    }
pd.DataFrame(rows).T.round(2)

<details>
<summary><b>คำอธิบายข้อ 2</b> (คลิกเพื่อเปิด)</summary>

ให้ดูว่า **ช่องว่าง** ระหว่างสองคอลัมน์เปลี่ยนอย่างไร

เมื่อมี lat/lon แบบจำลองเรียนรู้ได้ว่า "ตำแหน่งนี้มักมีค่าระดับนี้" ซึ่งช่วยมากเมื่อสถานีเดิมอยู่ในชุดฝึก (แบบสุ่ม)
แต่ช่วยน้อยลงเมื่อสถานีทดสอบเป็นที่ใหม่ ถ้าตัดออกแล้วช่องว่างแคบลง แสดงว่าส่วนหนึ่งของความแม่นยำแบบสุ่มมาจากการจำตำแหน่ง

</details>

### ข้อ 3 (ฝึกคิด) — ภาคใดทำนายยากที่สุด
ใช้ค่าทำนายแบบกันสถานีของ Random Forest คำนวณ RMSE แยกรายภาค

In [ ]:
# เขียนโค้ดของคุณที่นี่
# แนวทาง: สร้างตารางที่มี region, ค่าจริง y และค่าทำนาย pred[("Random Forest", "station")] แล้ว groupby

In [ ]:
#@title เฉลยข้อ 3 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
check = pd.DataFrame({"region": pairs["region"], "obs": y, "pred": pred[("Random Forest", "station")]})
check.groupby("region").apply(
    lambda d: pd.Series({"n": len(d), "mean PM2.5": d["obs"].mean(), "RMSE": np.sqrt(np.mean((d["obs"] - d["pred"]) ** 2))})
).round(1)

<details>
<summary><b>คำอธิบายข้อ 3</b> (คลิกเพื่อเปิด)</summary>

RMSE มักสูงในภาคที่ค่า PM2.5 สูงและแปรปรวนมาก จึงควรดูคู่กับค่าเฉลี่ยของภาคนั้น
RMSE 15 µg/m³ มีความหมายต่างกันมากระหว่างภาคที่ค่าเฉลี่ย 30 กับภาคที่ค่าเฉลี่ย 90

อีกปัจจัยคือจำนวนสถานี ภาคที่มีสถานีน้อย เมื่อกันออกไปทดสอบ แบบจำลองแทบไม่เหลือตัวอย่างจากภาคนั้นให้เรียนรู้

</details>

### ข้อ 4 (ฝึกตีความ)
เพื่อนร่วมชั้นรายงานว่า "แบบจำลองของเรามี $R^2$ = 0.85 จึงใช้ทำแผนที่ PM2.5 ทั้งประเทศได้อย่างน่าเชื่อถือ"
โดยใช้การแบ่งข้อมูลแบบสุ่ม ให้เขียนข้อโต้แย้ง 3–4 ประโยค

<details>
<summary><b>แนวคำตอบข้อ 4</b> (คลิกเพื่อเปิด)</summary>

การแบ่งแบบสุ่มทำให้วันต่าง ๆ ของสถานีเดียวกันอยู่ทั้งในชุดฝึกและชุดทดสอบ แบบจำลองจึงถูกทดสอบในตำแหน่งที่เคยเห็นแล้ว
ค่า $R^2$ นั้นบอกความสามารถในการเติมวันที่หายของสถานีเดิม ไม่ใช่การทำนายในพื้นที่ที่ไม่มีสถานี ซึ่งเป็นพื้นที่ส่วนใหญ่ของแผนที่
ควรรายงานผลแบบกันสถานีควบคู่กัน และระบุว่าความแม่นยำจะลดลงอีกในพื้นที่ที่ห่างจากสถานีหรือมีลักษณะต่างจากที่ตั้งสถานี เช่น ภูเขาและป่า

</details>

### ต่อยอดสำหรับ ป.โท
งานวิจัย AOD–PM2.5 ส่วนใหญ่เพิ่มตัวแปรอุตุนิยมวิทยา เช่น ความสูงชั้นขอบเขตบรรยากาศ (boundary layer height)
ความชื้นสัมพัทธ์ อุณหภูมิ และความเร็วลม จาก ERA5 ให้ออกแบบว่าจะเพิ่มตัวแปรเหล่านี้เข้าในตาราง `pairs` อย่างไร
และคาดว่าตัวแปรใดจะช่วยมากที่สุด เพราะอะไร

*แนวทาง* — ใน GEE ใช้ `ECMWF/ERA5_LAND/HOURLY` หรือ `ECMWF/ERA5/HOURLY` เลือกชั่วโมงใกล้เวลาดาวเทียมผ่าน แล้วดึงค่าที่สถานีแบบเดียวกับ AOD ใน Notebook 00
boundary layer height มักสำคัญที่สุด เพราะกำหนดว่าละอองลอยปริมาณเท่าเดิมถูกเจือจางในอากาศหนาเท่าใด
ข้อควรระวังคือ ERA5 มีความละเอียดหยาบกว่า AOD มาก (ราว 9–31 km)

**ตัวอย่างจากงานวิจัยจริง** — Bagheri (2022) ทำแผนที่ PM2.5 รายวันของกรุงเตหะรานจาก MAIAC AOD ([โค้ด](https://github.com/hbagh/AOD-PM2.5))
ใช้ตัวแปร 14 ตัว รวม boundary layer height, อุณหภูมิ, ความชื้น, ลม จาก ERA5 และปรับตัวแปรด้วยหลักกายภาพก่อนป้อนแบบจำลองสองขั้น

$$nAOD = \frac{AOD}{BLH} \qquad\qquad PM_c = \frac{PM_{2.5}}{1 - RH}$$

- $nAOD$ หาร AOD ด้วยความสูงชั้นขอบเขตบรรยากาศ (km) เพื่อประมาณความเข้มข้นของละอองลอย **ใกล้พื้น** แทนทั้งคอลัมน์
- $PM_c$ ปรับ PM2.5 ด้วยความชื้นสัมพัทธ์ (0–1) เพื่อให้เทียบกับ AOD ซึ่งวัดอนุภาคที่พองตัวจากความชื้นได้

ลองคิดว่าการปรับสองขั้นนี้ตอบปัญหาข้อใดใน "เหตุผลที่ AOD ไม่เท่ากับ PM2.5" ของ Notebook 01